<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SparkSQL_Intro_SLU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark SQL — Intro

## Welcome! 👋

In this notebook, we’ll walk through the basics of working with Spark SQL.

The goal is simple: by the end, you should feel comfortable loading data, exploring it, and writing both DataFrame and SQL-style queries.

We’ll keep things practical and build intuition step by step.

## 1. Creating a Spark Session

First, we need to start a Spark session.

👉 This is the entry point to Spark — everything we do will go through this session.

If you're running this in an environment like Colab and Spark is not available, you may need to install PySpark first:


In [3]:
# Uncomment this by removing in the following line the # symbol if the below cell says Spark is undefined and run the cell with the uncommented line
# !pip install pyspark

To create a Spark session, we first need to import `SparkSession` from PySpark:

In [4]:
from pyspark.sql import SparkSession

Now we create the session and give it a name (this is just a label for your application):

In [5]:
spark = SparkSession.builder \
    .appName("Spark SQL Intro") \
    .getOrCreate()

## 2. Creating a DataFrame

Let’s start with a simple example: creating a DataFrame from a list.

This helps us understand how Spark structures data.


In [6]:
data = [
    (1, "Alice", 25),
    (2, "Bob", 30),
    (3, "Charlie", 35)
]

columns = ["id", "name", "age"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



👉 Note how Spark automatically organizes the data into columns.

---

## 3. Understanding the Schema

In Spark, the **schema** is very important.

It tells us:

* what columns exist
* the data types
* how the data is structured

Let’s inspect it:


In [9]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



👉 Always check the schema when something looks off.

---

## 4. Selecting Columns

Now let’s pick only the columns we care about.

This is similar to a SQL `SELECT`.


In [11]:
df.select("name", "age").show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



Now, let’s go one step further.

Sometimes we don’t just want to select columns—we want to **compute new values using them**.

👉 This is where **column expressions** come in.

A *column expression* is how we tell Spark to apply logic to columns (instead of just returning them as-is).

You’ll use column expressions when you want to:

* do calculations (e.g., `age + 10`)
* compare values (e.g., `age > 30`)
* create new derived columns

Think of it like writing formulas on top of your data.

To use column expressions, we need to import `col` from PySpark:

In [10]:
from pyspark.sql.functions import col

Now we can create new columns:

In [12]:
df.select(
    "name",
    (col("age") + 10).alias("age_plus_10")
).show()

+-------+-----------+
|   name|age_plus_10|
+-------+-----------+
|  Alice|         35|
|    Bob|         40|
|Charlie|         45|
+-------+-----------+



👉 Think of this as transforming your data while selecting it.


---

### Alternative syntax: using `df.column_name`

Instead of using `col("age")`, you can also reference columns like this:




In [36]:
df.select(df.name, df.age).show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



And for expressions:

In [37]:
df.select((df.age + 10).alias("age_plus_10")).show()

+-----------+
|age_plus_10|
+-----------+
|         35|
|         40|
|         45|
+-----------+



👉 This is just another way to write column expressions.

### What is a Column object?

In Spark, most column operations return a `Column` object.

Let’s verify this:

In [38]:
from pyspark.sql import Column
from pyspark.sql.functions import upper

type(df.age) == type(upper(df.age)) == type(df.age.isNull())

True

In [39]:
df.age

Column<'age'>

👉 All of these return `Column` objects.

This is important because:

* Spark builds a **query plan**, not immediate results
* operations are **lazy** (they run only when you call an action like `show()`)

👉 Think of `Column` as a *recipe*, not the final result.


## 5. Inspecting Data (show, take, collect, tail)

When working with Spark, you’ll often want to quickly look at your data.

👉 These are called **actions**, they actually trigger computation and return results.

### `show()`

* If you don’t pass a number, Spark shows **20 rows by default**
* The number inside `show(n)` controls **how many rows you want to preview**

This is the most common way to quickly inspect data.




In [42]:
df.show(1)

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
+---+-----+---+
only showing top 1 row



You can also display rows vertically (very useful for wide tables):

In [43]:
df.show(5, vertical=True)

-RECORD 0-------
 id   | 1       
 name | Alice   
 age  | 25      
-RECORD 1-------
 id   | 2       
 name | Bob     
 age  | 30      
-RECORD 2-------
 id   | 3       
 name | Charlie 
 age  | 35      



👉 Each row is printed top-to-bottom instead of left-to-right.

____


### `take(n)`

👉 Returns the first *n* rows as a **Python list of Row objects**.

This is useful when:

* you want to programmatically access the data
* you don’t just want to print it

In [46]:
df.take(2)

[Row(id=1, name='Alice', age=25), Row(id=2, name='Bob', age=30)]


### `collect()`


👉 Brings **all rows** from Spark to your notebook (driver).

This means:

* Spark executes the full computation
* all data is loaded into memory locally

⚠️ Be careful:

* This is safe only for **small datasets**
* On large datasets, this can crash your notebook

👉 Rule of thumb: use `show()` instead unless you really need all the data.


In [47]:
df.collect()

[Row(id=1, name='Alice', age=25),
 Row(id=2, name='Bob', age=30),
 Row(id=3, name='Charlie', age=35)]

### `tail(n)`

👉 Returns the last n rows.

This is less commonly used but helpful when:

* you want to inspect the end of a dataset


In [48]:
df.tail(3)

[Row(id=1, name='Alice', age=25),
 Row(id=2, name='Bob', age=30),
 Row(id=3, name='Charlie', age=35)]



## 6. Filtering Rows

Next, let’s filter the data.

This is like a SQL `WHERE` clause.

In [13]:
df.filter(col("age") > 28).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



You can also use a SQL-style string:

In [14]:
df.where("age > 28").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



## 7. Working with Null Values

Handling missing data is very common in real datasets.

### Checking for nulls

 `isNull()` returns a Column expression.

In [53]:
data_with_nulls = [
    (1, "Alice", 25),
    (2, None, 30),
    (3, "Charlie", None)
]

columns = ["id", "name", "age"]

df_with_null = spark.createDataFrame(data_with_nulls, columns)
df_with_null.show()

+---+-------+----+
| id|   name| age|
+---+-------+----+
|  1|  Alice|  25|
|  2|   NULL|  30|
|  3|Charlie|NULL|
+---+-------+----+



In [54]:
df_with_null.age.isNull()

Column<'isNull(age)'>

In [55]:
df_with_null.filter(df_with_null.age.isNull()).show()

+---+-------+----+
| id|   name| age|
+---+-------+----+
|  3|Charlie|NULL|
+---+-------+----+



### Filtering non-null values


In [56]:
df_with_null.filter(df_with_null.age.isNotNull()).show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
|  2| NULL| 30|
+---+-----+---+



### Filling null values

In [57]:
df_with_null.fillna({"age": 0}).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|   NULL| 30|
|  3|Charlie|  0|
+---+-------+---+



Replaces null values with a default value.

**How this works**

* `fillna()` expects either:

  1. a **dictionary** → `{column_name: value}`
  2. a **single value** → applied to all compatible columns

In the example above:

```python
{"age": 0}
```

* `"age"` → the column we want to modify
* `0` → the value that will replace `null`

👉 So we are saying: *“In the `age` column, replace nulls with 0.”*


### Filling multiple columns

You can pass multiple columns in the dictionary:


In [59]:
df_with_null.fillna({
    "age": 0,
    "name": "unknown"
}).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|unknown| 30|
|  3|Charlie|  0|
+---+-------+---+




### Using a single value
This replaces nulls in **all numeric columns** with `0` :

In [61]:
df_with_null.fillna(0).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|   NULL| 30|
|  3|Charlie|  0|
+---+-------+---+



### Dropping null values
Removes rows that contain null values.

More specifically:

* by default, it removes rows where **any column is null**


In [62]:
df_with_null.dropna().show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
+---+-----+---+



You can control this behavior, This removes rows only if `age` is null.:

In [64]:
df_with_null.dropna(subset=["age"]).show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
|  2| NULL| 30|
+---+-----+---+



 Summary:

* `isNull()` → check for nulls
* `fillna()` → replace nulls
* `dropna()` → remove rows with nulls

These are essential when cleaning real-world data.




## 8. Reading Data (CSV)

Let’s move to a more realistic scenario: reading data from a file.

CSV files are common, but we need to tell Spark how to interpret them.


In [17]:
df_csv = spark.read.option("header", True).option("inferSchema", True) \
.csv("/content/spark_sql_intro_data/people.csv")

df_csv.show()

+---+-----+---+-------+----------+------+
| id| name|age|country|event_date|salary|
+---+-----+---+-------+----------+------+
|  1|  Ana| 28|     ES|2025-01-15|  4200|
|  2|  Bob| 34|     US|2025-01-16|  5100|
|  3|Carla| 25|     ES|2025-02-01|  3900|
|  4|Diego| 41|     MX|2025-02-10|  6100|
|  5|  Eva| 31|     US|2025-03-05|  4700|
+---+-----+---+-------+----------+------+



👉 `header=True` tells Spark the first row has column names.
👉 `inferSchema=True` lets Spark guess data types.

---

## 9. Reading Data (JSON)

JSON is often nested, and Spark handles it nicely.



In [18]:
df_json = spark.read.json("/content/spark_sql_intro_data/people_nested.json")
df_json.show(truncate=False)

+------------------+---+-----+-----------------------------+--------+
|hobbies           |id |name |orders                       |scores  |
+------------------+---+-----+-----------------------------+--------+
|[reading, cycling]|1  |Ana  |[{120.5, A100}, {80.0, A101}]|{85, 90}|
|[gaming, running] |2  |Bob  |[{45.0, B200}]               |{88, 75}|
|[music]           |3  |Carla|[]                           |{91, 95}|
+------------------+---+-----+-----------------------------+--------+




👉 JSON is where Spark really shines because of nested structures.

---

## 10. Grouping and Aggregation

Let’s summarize our data.

For example:

* How many rows do we have per group?
* What is the average value?

We need to import aggregation functions:

In [19]:
from pyspark.sql.functions import count, avg

Now we can group and aggregate:


In [20]:
df.groupBy("age").agg(
    count("*").alias("count"),
    avg("age").alias("avg_age")
).show()

+---+-----+-------+
|age|count|avg_age|
+---+-----+-------+
| 25|    1|   25.0|
| 35|    1|   35.0|
| 30|    1|   30.0|
+---+-----+-------+



## 11. Renaming Columns with `withColumnRenamed`

Sometimes column names are not very clear, or we want to make them easier to use.

For that, Spark gives us `withColumnRenamed()`.

👉 This function lets us rename an existing column.

For example, if we want to rename `age` to `customer_age`:

In [66]:
df.withColumnRenamed("age", "customer_age")

DataFrame[id: bigint, name: string, customer_age: bigint]

In [ ]:
df

This is useful when:

* a column name is too generic
* you want clearer names before a join
* you want your output to be easier to read

👉 `withColumnRenamed()` changes the column name, but it does not change the actual values inside the column.


## 12. Temporary Views and SQL

So far, we’ve been using the DataFrame API (functions like `select`, `filter`, `groupBy`).

👉 But Spark also lets us write queries using **SQL syntax**, just like in a database.

To do that, we need something called a **temporary view**.

### What is a temporary view?

A temporary view is basically a **name we give to a DataFrame so we can query it using SQL**.

👉 You can think of it like creating a temporary table in memory.

It does NOT save data to disk, and it only exists while your Spark session is running.

---

### Creating a temporary view

We register our DataFrame as a view like this:

In [21]:
df.createOrReplaceTempView("people")

👉 Now Spark knows: “there is a table called `people` that I can query with SQL”.

### Querying using SQL

Instead of using functions like `groupBy`, we can now write a SQL query as a string:


In [22]:
spark.sql("""
SELECT age, COUNT(*) as total
FROM people
GROUP BY age
""").show()

+---+-----+
|age|total|
+---+-----+
| 25|    1|
| 35|    1|
| 30|    1|
+---+-----+




👉 This is what we mean by “querying with SQL”:

* writing SQL text (`SELECT`, `FROM`, `GROUP BY`)
* instead of chaining DataFrame functions

### Why is this useful?

* It’s easier for people who already know SQL
* It makes complex queries more readable
* You can switch between DataFrame API and SQL anytime

👉 Under the hood, Spark runs both in the same engine.

---

### Important notes about temporary views

* They are **session-scoped** (they disappear when Spark stops)
* They are **not saved permanently**
* You can replace them anytime using `createOrReplaceTempView`

---

👉 This is great for analysts and for mixing SQL with Python in the same workflow.


## 13. Working with Dates

Date columns are very common in analytics.

For example, you might want to answer questions like:

* how many events happened in each month?
* what year does this record belong to?
* how do I filter rows by date?

Before doing that, it helps to make sure Spark understands the column as a real **date type**.

Right now, a date coming from a CSV is often just a string.
So first, we convert it.

To work with dates, we need to import a few functions:


In [67]:
from pyspark.sql.functions import to_date, year, month

Here is what each one does:

* `to_date()` converts a string column into a date column
* `year()` extracts the year from a date
* `month()` extracts the month number from a date

Now let’s apply them:


In [68]:
df_dates = (
    df_csv
    .withColumn("event_date", to_date("event_date"))
    .withColumn("event_year", year("event_date"))
    .withColumn("event_month", month("event_date"))
)

df_dates.show()
df_dates.printSchema()

+---+-----+---+-------+----------+------+----------+-----------+
| id| name|age|country|event_date|salary|event_year|event_month|
+---+-----+---+-------+----------+------+----------+-----------+
|  1|  Ana| 28|     ES|2025-01-15|  4200|      2025|          1|
|  2|  Bob| 34|     US|2025-01-16|  5100|      2025|          1|
|  3|Carla| 25|     ES|2025-02-01|  3900|      2025|          2|
|  4|Diego| 41|     MX|2025-02-10|  6100|      2025|          2|
|  5|  Eva| 31|     US|2025-03-05|  4700|      2025|          3|
+---+-----+---+-------+----------+------+----------+-----------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- salary: integer (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: integer (nullable = true)



`withColumn()` is used here to create new columns.

In this example:

* we replace `event_date` with a real date type
* we create `event_year`
* we create `event_month`

After this, Spark understands the column as a date, which makes date-based analysis much easier.


## 14. Working with Nested Data

Real-world data (especially JSON) is often nested.

For example, you might have:

* arrays (lists)
* maps (key-value pairs)

👉 Let’s look at a simple example of an array column:

```python
# Example structure

# name | hobbies
# -------------------------
# Ana  | ["reading", "sports"]
# Bob  | ["gaming"]
```

Here, each row has a **list inside a single column**.

Spark keeps this as one column, not multiple rows.

### Why is this a problem?

If we want to analyze hobbies (e.g., count how many times each hobby appears),
we need each hobby to be on its own row.

👉 But right now, they are grouped inside a list.


To fix this, we use `explode()`.

👉 `explode()` takes a column that contains a list (array) and **"unpacks" it into multiple rows**.

So this:

```
Ana  | ["reading", "sports"]
```

Becomes:

```
Ana  | reading
Ana  | sports
```

👉 One row becomes multiple rows — one per element in the array.

---

Think of `explode()` like:
👉 "Take everything inside the list and spread it out into separate rows".

We’ll see this in action in the next section.




## 15. Exploding Arrays

To use `explode`, we first need to import it:


In [23]:
from pyspark.sql.functions import explode

Look how the data looks like before exploding

In [25]:
df_json.select(
    "name",
   "hobbies"
).show()

+-----+------------------+
| name|           hobbies|
+-----+------------------+
|  Ana|[reading, cycling]|
|  Bob| [gaming, running]|
|Carla|           [music]|
+-----+------------------+



After using explode, each element in the array becomes its own row:

In [24]:
df_json.select(
    "name",
    explode("hobbies").alias("hobby")
).show()

+-----+-------+
| name|  hobby|
+-----+-------+
|  Ana|reading|
|  Ana|cycling|
|  Bob| gaming|
|  Bob|running|
|Carla|  music|
+-----+-------+



## 16. Exploding Maps

Before we explode a map, let’s quickly understand what a **map** is.

👉 A *map* is a collection of **key-value pairs** (like a dictionary in Python).

Example:

```python
# Example structure
# name | scores
# ------------------------------
# Ana  | {"math": 90, "english": 85}
# Bob  | {"math": 75, "english": 88}
```

Here:

* `math`, `english` → are the **keys**
* `90`, `85` → are the **values**


---

### Why explode a map?

Just like arrays, maps are stored inside a single column.

If we want to analyze them (for example, compare scores by subject),
we need each key-value pair to be on its own row.

---

### Exploding a map

When we use `explode()` on a map, Spark creates two columns:

* `key`
* `value`


In [28]:
from pyspark.sql import Row

data = [
    Row(name="Ana", scores={"math": 90, "english": 85}),
    Row(name="Bob", scores={"math": 75, "english": 88}),
]

df_map = spark.createDataFrame(data)

df_map.show(truncate=False)
df_map.printSchema()

+----+---------------------------+
|name|scores                     |
+----+---------------------------+
|Ana |{english -> 85, math -> 90}|
|Bob |{english -> 88, math -> 75}|
+----+---------------------------+

root
 |-- name: string (nullable = true)
 |-- scores: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)



Exploding scores columns creates a key and value columns

In [29]:
df_map.select(
    "name",
    explode("scores")
).show()

+----+-------+-----+
|name|    key|value|
+----+-------+-----+
| Ana|english|   85|
| Ana|   math|   90|
| Bob|english|   88|
| Bob|   math|   75|
+----+-------+-----+



## 17. SQL Version of `explode`

So far, we have used `explode()` with the DataFrame API.

But if you want to stay in SQL, Spark also lets you do the same thing there.

For this, Spark uses `LATERAL VIEW` together with `EXPLODE`.

### What does `LATERAL VIEW` do?

`LATERAL VIEW` lets us take the result of a function like `EXPLODE` and attach it back to each original row.

👉 In simpler words: it lets SQL “expand” arrays or maps into rows.

---

### SQL example: exploding an array

First, we register the JSON DataFrame as a temporary view:


In [74]:
df_json.createOrReplaceTempView("people_nested")

Now we can write a SQL query:

In [73]:
spark.sql("""
SELECT name, hobby
FROM people_nested
LATERAL VIEW explode(hobbies) exploded_hobbies AS hobby
""").show()

{"ts": "2026-04-07 22:52:15.670", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `hobbies` cannot be resolved. Did you mean one of the following? [`name`, `scores`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o26.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `hobbies` cannot be resolved. Did you mean one of the following? [`name`, `scores`]. SQLSTATE: 42703; line 4 pos 21;\n'Project ['name, 'hobby]\n+- 'Generate 'explode('hobbies), false, exploded_hobbies, ['hobby]\n   +- SubqueryAlias people_nested\n      +- View (`people_nested`, [name#235, scores#236])\n         +- LogicalRDD [name#235, scores#236], false\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAtt

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `hobbies` cannot be resolved. Did you mean one of the following? [`name`, `scores`]. SQLSTATE: 42703; line 4 pos 21;
'Project ['name, 'hobby]
+- 'Generate 'explode('hobbies), false, exploded_hobbies, ['hobby]
   +- SubqueryAlias people_nested
      +- View (`people_nested`, [name#235, scores#236])
         +- LogicalRDD [name#235, scores#236], false


Here’s what is happening:

* `explode(hobbies)` takes the hobbies array
* `LATERAL VIEW` expands it into rows
* `AS hobby` gives a name to the exploded value

👉 This gives us one row per hobby.

### SQL example: exploding a map

We can do the same thing with a map:

In [75]:
df_map.createOrReplaceTempView("people_map")
spark.sql("""
SELECT name, subject, score
FROM people_map
LATERAL VIEW explode(scores) exploded_scores AS subject, score
""").show()

+----+-------+-----+
|name|subject|score|
+----+-------+-----+
| Ana|english|   85|
| Ana|   math|   90|
| Bob|english|   88|
| Bob|   math|   75|
+----+-------+-----+



Here:

* `subject` is the key
* `score` is the value

## 18. Working with Structs (and how to explode them)

Before we try to explode anything, we need to understand the data type we’re dealing with.

👉 In our dataset, `scores` is a **struct**, not a map.

### What is a struct?

A *struct* is like a **fixed schema object** with named fields (similar to a row inside a row).

Example:

```python
# name | scores
# ------------------------------
# Ana  | {math: 90, english: 85}
# Bob  | {math: 75, english: 88}
```

Here:

* `math` and `english` are **fields (columns inside the struct)**
* Values are accessed using dot notation

____

### Important ⚠️

👉 You **CANNOT use `explode()` on a struct**

That’s why this fails:







In [34]:
df_json.select(
    "name",
    explode("scores")   # ❌ ERROR
).show()

{"ts": "2026-04-07 22:10:58.055", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"explode(scores)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"scores\" has the type \"STRUCT<english: BIGINT, math: BIGINT>\". SQLSTATE: 42K09", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "explode", "errorClass": "DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o101.select.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"explode(scores)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"scores\" has the type \"STRUCT<english: BIGINT, math: BIGINT>\". SQLSTATE: 42K09;\n'Project [name#154, unresolvedalias(explode(scores

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "explode(scores)" due to data type mismatch: The first parameter requires the ("ARRAY" or "MAP") type, however "scores" has the type "STRUCT<english: BIGINT, math: BIGINT>". SQLSTATE: 42K09;
'Project [name#154, unresolvedalias(explode(scores#156))]
+- Relation [hobbies#152,id#153L,name#154,orders#155,scores#156] json


In [35]:
df_json.printSchema()

root
 |-- hobbies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- order_id: string (nullable = true)
 |-- scores: struct (nullable = true)
 |    |-- english: long (nullable = true)
 |    |-- math: long (nullable = true)




Because `explode()` only works with:

* arrays
* maps

## How to work with structs

### 1. Access fields directly

You can select struct fields using dot notation:

In [33]:
df_json.select(
    "name",
    "scores.math",
    "scores.english"
).show()

+-----+----+-------+
| name|math|english|
+-----+----+-------+
|  Ana|  90|     85|
|  Bob|  75|     88|
|Carla|  95|     91|
+-----+----+-------+



👉 This simply flattens the struct into columns.

---

### 2. Expand the struct automatically

If you want all fields inside the struct:


In [32]:
df_json.select(
    "name",
    "scores.*"
).show()

+-----+-------+----+
| name|english|math|
+-----+-------+----+
|  Ana|     85|  90|
|  Bob|     88|  75|
|Carla|     91|  95|
+-----+-------+----+




## Key takeaway

* **Struct = fixed fields → use dot notation (`scores.math`)**
* **Map = dynamic key-value → can use `explode()` directly**
* To explode a struct → **convert it into a map first**
